# Hierarchical mergers of binary black holes

All the relevant information for the project are to be found in the pdf document present in the repo.
Note that you are assigned to project 2 (as the title said).

## Datasets 

Datasets are stored on Google Drive (link and description in the pdf document)

### Contacts

* Giuliano Iorio <giuliano.iorio@unipd.it>


## Libraries 

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import patchworklib as pw
from support_code.EDA_utils import dataset_drift_report
from support_code.plot_utils import plot_joint_threshold_split
pw.overwrite_axisgrid() #When you use pw.load_seagorngrid, 'overwrite_axisgrid' should be executed　in advance


<Figure size 100x100 with 0 Axes>

## Data load

Here we load the different files containing all the data that we will use for this project. We will merge all data in to one large file adding flags to the nature of the system and to the metallicitty so we don't loose information. This will make it easier in order to make an exploratory analysis

In [63]:
# uncomment and run this code block to read and combine all nth_generation.txt files into a single DataFrame

#def read_nth_generation(path):
#    # read and repair header (merge tokens that start with '(' into previous token)
#    with open(path, 'r') as f:
#        header = f.readline().strip()
#    tokens = header.split()
#    names = []
#    for tok in tokens:
#        if tok.startswith('(') and names:
#            names[-1] = names[-1] + ' ' + tok
#        else:
#            names.append(tok)
#    # read remaining rows using whitespace splitting and assign repaired names
#    df = pd.read_csv(path, sep='\s+', header=None, names=names, skiprows=1, comment='#')
#    return df
#
#base_path = Path('fastcluster_comp_physA')
#all_data = []
#for filepath in base_path.glob('**/*/Dyn/*/nth_generation.txt'):
#    sys_origin = filepath.parts[-4].split('_')[0]
#    mettalictty = float(filepath.parts[-2])
#    df = read_nth_generation(filepath)
#    df['sys'] = sys_origin
#    df['met'] = mettalictty
#    all_data.append(df)
#
#df_final = pd.concat(all_data, ignore_index=True, sort=False)

Now we drop the columns that have no useful information for our classification task

In [2]:
# after running the above code block once, you can comment it out and read the combined CSV file directly in the future, uncomment the following lines to drop unnecessary columns,
#  convert identifier to category, and save the cleaned DataFrame to a new CSV file. You can also inspect the columns and data types before saving.:
#print(df_final.shape)
#df_final = df_final.dropna()  # drop rows with any NaN values
#print(df_final.shape)
#df_final = df_final[df_final['c9:(tDF+min(t12,t3bb))/Myr'] < 13.9*1000]  # filter out rows where this column is not positive
#print(df_final.shape)
##df_final.drop(columns=['c5:theta1', 'c6:theta2', 'c7:SMA(Rsun)', 'c8:ecc','c10:SMAfin(cm)', 'c11:eccfin',
##       'c12:tpeters/Myr', 'c14:vkick/kms', 'c18:flag1', 'c19:flag2', 'c20:flag3', 'c21:flagSN', 'c22:flag_exch',
##       'c23:flag_t3bb', 'c24:flag_evap','c26:ecc(10Hz)'], inplace=True)
#df_final['c0:identifier'] = df_final['c0:identifier'].astype('category')
#
#df_final.columns = [col.split('/')[0] for col in df_final.columns]
#df_final.columns
#df_final.to_csv('combined_nth_generation.csv', index=False)
df_final = pd.read_csv('combined_nth_generation.csv')

## Exploratory Data Analysis (EDA)

Now we will perform an exploration of the different parameters to detect some outliers and clean the datasets in order to make sure we have consistent measurements. 

Now we separate the data into three different dataframes according to the type of system they belong to, this will make it easier incase we want to analyze the data in a same group

In [65]:
df_GC = df_final[df_final['sys'] == 'GC']
df_NSC = df_final[df_final['sys'] == 'NSC']
df_YSC = df_final[df_final['sys'] == 'YSC']

In [66]:
sample_size = 200000
df_GC_sampled = df_GC.sample(frac=sample_size/df_GC.shape[0], random_state=42)
df_NSC_sampled = df_NSC.sample(frac= sample_size/df_NSC.shape[0], random_state=42)

In [67]:
dfs_NSC = {
    
    "NSC (Orig)": df_NSC,
    
    "NSC (Samp)": df_NSC_sampled
}
dfs_GC = {
    "GC (Orig)": df_GC,
    "GC (Samp)": df_GC_sampled
}

In [68]:
text_report, stats_table, drift_table = dataset_drift_report(
    dfs_NSC,
    quantile_cut=1,
    drift_threshold=5,
    columns_to_compare=['c1:M1', 'c2:M2', 'c3:chi1', 'c4:chi2','c9:(tDF+min(t12,t3bb))', 'c13:(ngen (tDF+t3bb+tpeters))','c15:mrem', 'c16:arem', 'c17:vesc', 'c25:Mtot']
)
print(text_report)


Sample fractions:0.0714259439742038
DATASET DRIFT REPORT
Base Dataset: NSC (Orig)
Outlier filter: 100% Quantile
Threshold: 5%
=============================== ============== ========= =========== ======== ========
            Column               Comp vs Base   Mean Δ%   Median Δ%   Std Δ%   Status 
=============================== ============== ========= =========== ======== ========
 c13:(ngen (tDF+t3bb+tpeters))    NSC (Samp)     0.28%     -0.13%     0.61%      OK   
           c15:mrem               NSC (Samp)    -0.05%     -0.08%     -0.07%     OK   
           c16:arem               NSC (Samp)    -0.01%     -0.00%     0.51%      OK   
           c17:vesc               NSC (Samp)    -0.01%     -0.07%     -0.19%     OK   
             c1:M1                NSC (Samp)    -0.02%     -0.11%     0.32%      OK   
           c25:Mtot               NSC (Samp)    -0.18%      0.04%     -0.49%     OK   
             c2:M2                NSC (Samp)    -0.09%      0.02%     -0.07%     OK   
    

In [69]:
text_report, stats_table, drift_table = dataset_drift_report(
    dfs_GC,
    quantile_cut=1,
    drift_threshold=5,
    columns_to_compare=['c1:M1', 'c2:M2', 'c3:chi1', 'c4:chi2','c9:(tDF+min(t12,t3bb))', 'c13:(ngen (tDF+t3bb+tpeters))','c15:mrem', 'c16:arem', 'c17:vesc', 'c25:Mtot']
)
print(text_report)

Sample fractions:0.45625433441617697
DATASET DRIFT REPORT
Base Dataset: GC (Orig)
Outlier filter: 100% Quantile
Threshold: 5%
=============================== ============== ========= =========== ======== ========
            Column               Comp vs Base   Mean Δ%   Median Δ%   Std Δ%   Status 
=============================== ============== ========= =========== ======== ========
 c13:(ngen (tDF+t3bb+tpeters))    GC (Samp)      0.68%     -0.21%     1.02%      OK   
           c15:mrem               GC (Samp)      0.20%      0.25%     0.50%      OK   
           c16:arem               GC (Samp)     -0.02%      0.00%     1.01%      OK   
           c17:vesc               GC (Samp)      0.09%      0.08%     0.23%      OK   
             c1:M1                GC (Samp)      0.20%      0.27%     0.48%      OK   
           c25:Mtot               GC (Samp)      0.32%      0.23%     0.65%      OK   
             c2:M2                GC (Samp)      0.23%      0.24%     0.45%      OK   
    

Since we have shown that we can use a reduced versio of the data without compromising the statistics of the dataset by much we can therefore create the dataframes 

In [70]:
GC_cut = df_GC_sampled.select_dtypes(include="number").quantile(.9)
GC_q = df_GC_sampled.where(df_GC_sampled.le(GC_cut))
NSC_cut = df_NSC_sampled.select_dtypes(include="number").quantile(.9)
NSC_q = df_NSC_sampled.where(df_NSC_sampled.le(NSC_cut))

In [71]:
#from ydata_profiling import ProfileReport

In [72]:
#profile_GC = ProfileReport(df_GC, title="Profiling Report for GC", explorative=True)
#profile_GC.to_file("GC_report.html")
#profile_YSC = ProfileReport(df_YSC, title="Profiling Report for YSC", explorative=True)
#profile_YSC.to_file("YSC_report.html")
#profile_NSC = ProfileReport(df_NSC, title="Profiling Report for NSC", explorative=True)
#profile_NSC.to_file("NSC_report.html")
#profile_total = ProfileReport(df_final, title="Profiling Report for total dataset", explorative=True)
#profile_total.to_file("total_report.html")

After seeing the reports we notice that the values of the identifier column has values that are not unique so we will have to see if the rest of the columns has the same values and the entire row is duplicated or anything is else is happening

In [73]:
id = df_final['c0:identifier'].unique()
for i in id[:2]:
    subset = df_final[df_final['c0:identifier'] == i]
    if len(subset) > 1:
        for col in df_final.columns:
            if subset[col].nunique() > 1:
                print(f"Identifier {i} has {subset[col].nunique()} different values in column {col}")

Identifier 0 has 2 different values in column c1:M1
Identifier 0 has 2 different values in column c2:M2
Identifier 0 has 2 different values in column c3:chi1
Identifier 0 has 2 different values in column c4:chi2
Identifier 0 has 2 different values in column c5:theta1
Identifier 0 has 2 different values in column c6:theta2
Identifier 0 has 2 different values in column c7:SMA(Rsun)
Identifier 0 has 2 different values in column c8:ecc
Identifier 0 has 2 different values in column c9:(tDF+min(t12,t3bb))
Identifier 0 has 2 different values in column c10:SMAfin(cm)
Identifier 0 has 2 different values in column c11:eccfin
Identifier 0 has 2 different values in column c12:tpeters
Identifier 0 has 2 different values in column c13:(ngen (tDF+t3bb+tpeters))
Identifier 0 has 2 different values in column c14:vkick
Identifier 0 has 2 different values in column c15:mrem
Identifier 0 has 2 different values in column c16:arem
Identifier 0 has 2 different values in column c17:vesc
Identifier 0 has 2 dif

After executing the code we arrived to the conclusion that the rows with the same identifier are the same system after some number of generations, because we see that the only column that has the same value is the 'nGen' column

In [3]:
from itertools import combinations
from scipy.stats._continuous_distns import alpha
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os

def plot_joint_per_category(dfs, labels, x_col, y_col, hue_col,label_dict, quantile=1):
    # 1. Get all unique values for the category across all datasets
    unique_cats = sorted(pd.concat([d[hue_col] for d in dfs]).unique())
    
    # 2. Global limits (90th percentile) for consistent scale across all plots
    all_x = pd.concat([d[x_col] for d in dfs])
    all_y = pd.concat([d[y_col] for d in dfs])
    x_lim = all_x.quantile(quantile)
    y_lim = all_y.quantile(quantile)
    x_min, y_min = all_x.min(), all_y.min()

    colors = ["#3498db", "#e74c3c", "#2ecc71"] # Blue, Red, Green
    markers = ['o', 's', '^']

    # 3. Create one figure for each category
    for cat in unique_cats:
        os.makedirs(f"images/{hue_col}/{cat}", exist_ok=True)
        # Initialize the JointGrid for this specific category
        g = sns.JointGrid(height=6)
        
        for i, df in enumerate(dfs):
            # Filter the current dataset for the current category
            subset = df[df[hue_col] == cat]
            
            if subset.empty:
                continue
            num_samples = len(subset)
            # Sample for the scatter plot (Safety for millions of rows)
            scatter_alpha = max(0.01, min(0.5, 0.5 / (1 + np.log10(num_samples)) if num_samples > 0 else 0.5))
            
            # --- Center Plot ---
            sns.scatterplot(data=subset, x=x_col, y=y_col, ax=g.ax_joint,
                            color=colors[i], marker=markers[i], alpha=scatter_alpha, 
                            s=15, label=labels[i])
            
            sns.kdeplot(data=subset, x=x_col, y=y_col, ax=g.ax_joint,
                        color=colors[i], levels=4, alpha=1, warn_singular=True)
            
            # --- Marginal Histograms ---
            sns.histplot(data=subset, x=x_col, ax=g.ax_marg_x, color=colors[i],
                         element="step", fill=True, linewidth=1.5, common_norm=False,
                         binrange=(x_min, x_lim), alpha=0.2)
            
            sns.histplot(data=subset, y=y_col, ax=g.ax_marg_y, color=colors[i],
                         element="step", fill=True, linewidth=1.5, common_norm=False,
                         binrange=(y_min, y_lim), alpha=0.2)
        auto_x_min, auto_x_max = g.ax_joint.get_xlim()
        auto_y_min, auto_y_max = g.ax_joint.get_ylim()

        # 4. Final Formatting for this category plot
        g.ax_joint.set_xlim(auto_x_min, auto_x_max)
        g.ax_joint.set_ylim(auto_y_min, auto_y_max)
        g.ax_joint.set_xlabel(label_dict[x_col])
        g.ax_joint.set_ylabel(label_dict[y_col])
        
        # Set Title and Legend
        plt.suptitle(f"Feature: {label_dict[hue_col]} | Value: {cat}", y=1.02, fontsize=14, fontweight='bold')
        g.ax_joint.legend(title="Datasets", loc='lower right')
        
        plt.savefig(f"images/joint_plot_{x_col}_{y_col}.pdf", bbox_inches='tight')
        plt.close()

# python
label_dict = {
    'c0:identifier': r'$\mathrm{ID}$',
    'c1:M1': r'$M_1\,/\,M_\odot$',
    'c2:M2': r'$M_2\,/\,M_\odot$',
    'c3:chi1': r'$\chi_1$',
    'c4:chi2': r'$\chi_2$',
    'c9:(tDF+min(t12,t3bb))': r'$\left(t_{\mathrm{DF}}+\min\!\left(t_{12},t_{3\mathrm{bb}}\right)\right)/\mathrm{Myr}$',
    'c13:(ngen (tDF+t3bb+tpeters))': r'$\left(n_{\mathrm{gen}}\,(t_{\mathrm{DF}}+t_{3\mathrm{bb}}+t_{\mathrm{peters}})\right)/\mathrm{Myr}$',
    'c15:mrem': r'$m_{\mathrm{rem}}\,/\,M_\odot$',
    'c16:arem': r'$a_{\mathrm{rem}}$',
    'c17:vesc': r'$v_{\mathrm{esc}}\ (\mathrm{km\,s^{-1}})$',
    'c25:Mtot': r'$M_{\mathrm{tot}}\,/\,M_\odot$',
    'c27:Ngen': r'$N_{\mathrm{gen}}$',
    'sys': r'$\mathrm{system}$',
    'met': r'$Z$'
}



In [75]:
#combo_list = []
#for combo in combinations(['c1:M1', 'c2:M2', 'c3:chi1', 'c4:chi2','c9:(tDF+min(t12,t3bb))', 'c13:(ngen (tDF+t3bb+tpeters))','c15:mrem', 'c16:arem', 'c17:vesc', 'c25:Mtot'],2):
#    if combo not in combo_list and (combo[1], combo[0]) not in combo_list:
#        try:
#            plot_joint_per_category([GC_q, df_YSC, NSC_q], ['GC', 'YSC', 'NSC'], combo[0], combo[1], 'met',label_dict,1)
#            combo_list.append(combo)
#        except Exception as e:
#            print(f"Error plotting {combo[0]} vs {combo[1]}: {e}")
#            continue
#    else: 
#        continue

In [76]:
#combo_list = []
#for combo in combinations(['c1:M1', 'c2:M2', 'c3:chi1', 'c4:chi2','c9:(tDF+min(t12,t3bb))', 'c13:(ngen (tDF+t3bb+tpeters))','c15:mrem', 'c16:arem', 'c17:vesc', 'c25:Mtot'],2):
#    if combo not in combo_list and (combo[1], combo[0]) not in combo_list:
#        try:
#            plot_joint_per_category([GC_q, df_YSC, NSC_q], ['GC', 'YSC', 'NSC'], combo[0], combo[1], 'c3:chi1',label_dict,1)
#            combo_list.append(combo)
#        except Exception as e:
#            print(f"Error plotting {combo[0]} vs {combo[1]}: {e}")
#            continue
#    else: 
#        continue

In [77]:
#combo_list = []
#for combo in combinations(['c1:M1', 'c2:M2', 'c3:chi1', 'c4:chi2','c9:(tDF+min(t12,t3bb))', 'c13:(ngen (tDF+t3bb+tpeters))','c15:mrem', 'c16:arem', 'c17:vesc', 'c25:Mtot'],2):
#    if combo not in combo_list and (combo[1], combo[0]) not in combo_list:
#        try:
#            plot_joint_per_category([GC_q, df_YSC, NSC_q], ['GC', 'YSC', 'NSC'], combo[0], combo[1], 'c4:chi2',label_dict,1)
#            combo_list.append(combo)
#        except Exception as e:
#            print(f"Error plotting {combo[0]} vs {combo[1]}: {e}")
#            continue
#    else: 
#        continue

In [78]:
vals = sorted([0.0002, 0.008, 0.0012, 0.3])
filter = lambda x, a, b: (x >= a) & (x < b)

data = pd.concat([GC_q, df_YSC, NSC_q], ignore_index=True)
for i in range(len(vals)-1):
    data = data[data["c27:Ngen"] <= 2]
    data = data[filter(data["met"], vals[i], vals[i+1])]
    data.dropna(inplace=True)
    for sys in data['sys'].unique():
        subset = data[data['sys'] == sys]
        subset.dropna(inplace=True)
        print(f"System: {sys}, met range: [{vals[i]}, {vals[i+1]}), Ngen <= 2")
        print(subset.shape)


System: YSC, met range: [0.0002, 0.0012), Ngen <= 2
(3243, 30)


In [79]:
vals = sorted([0.0002, 0.008, 0.0012, 0.3])
filter = lambda x, a, b: (x >= a) & (x < b)

data = pd.concat([GC_q, df_YSC, NSC_q], ignore_index=True)
for i in range(len(vals)-1):
    data = data[data["c27:Ngen"] > 2]
    data = data[filter(data["met"], vals[i], vals[i+1])]
    data.dropna(inplace=True)
    for sys in data['sys'].unique():
        subset = data[data['sys'] == sys]
        subset.dropna(inplace=True)
        print(f"System: {sys}, met range: [{vals[i]}, {vals[i+1]}), Ngen > 2")
        print(subset.shape)

System: YSC, met range: [0.0002, 0.0012), Ngen > 2
(12, 30)


In [ ]:
from support_code.plot_utils import plot_joint_threshold_split
import glob
import re

combo_list = []
# Define the pattern to search for
# * is a wildcard that matches any characters
search_pattern = "images/*/threshold_split/*/joint_threshold_*_*.pdf"

# This regex extracts x_col and y_col from the specific filename format:
# joint_threshold_{x_col}_{y_col}.pdf
# [^_]+ matches characters that are not underscores
path_regex = r"joint_threshold_([^_]+)_([^_]+)\.pdf$"

data_list = []

# Loop through all files matching the pattern
for file_path in glob.glob(search_pattern):
    # Extract the filename from the full path
    file_name = file_path.split('/')[-1]
    
    # Use regex to find x_col and y_col
    match = re.search(path_regex, file_name)
    
    if match:
        x_val = match.group(1)
        y_val = match.group(2)
        combo_list.append((x_val, y_val))
    else:
        print(f"Filename {file_name} does not match the expected pattern.")
    


Saved to images/c27:Ngen/threshold_split/c1:M1/joint_threshold_c1:M1_c2:M2.pdf
Saved to images/c27:Ngen/threshold_split/c1:M1/joint_threshold_c1:M1_c3:chi1.pdf
Saved to images/c27:Ngen/threshold_split/c1:M1/joint_threshold_c1:M1_c4:chi2.pdf
Saved to images/c27:Ngen/threshold_split/c1:M1/joint_threshold_c1:M1_c9:(tDF+min(t12,t3bb)).pdf
Saved to images/c27:Ngen/threshold_split/c1:M1/joint_threshold_c1:M1_c13:(ngen (tDF+t3bb+tpeters)).pdf
Saved to images/c27:Ngen/threshold_split/c1:M1/joint_threshold_c1:M1_c15:mrem.pdf
Saved to images/c27:Ngen/threshold_split/c1:M1/joint_threshold_c1:M1_c16:arem.pdf
Saved to images/c27:Ngen/threshold_split/c1:M1/joint_threshold_c1:M1_c17:vesc.pdf
Saved to images/c27:Ngen/threshold_split/c1:M1/joint_threshold_c1:M1_c25:Mtot.pdf
Saved to images/c27:Ngen/threshold_split/c2:M2/joint_threshold_c2:M2_c3:chi1.pdf
Saved to images/c27:Ngen/threshold_split/c2:M2/joint_threshold_c2:M2_c4:chi2.pdf
Saved to images/c27:Ngen/threshold_split/c2:M2/joint_threshold_c2:M2_c

In [ ]:
for combo in combinations(['c1:M1', 'c2:M2', 'c3:chi1', 'c4:chi2','c9:(tDF+min(t12,t3bb))', 'c13:(ngen (tDF+t3bb+tpeters))','c15:mrem', 'c16:arem', 'c17:vesc', 'c25:Mtot'],2):#
    if combo not in combo_list and (combo[1], combo[0]) not in combo_list:
        try:
            pw.overwrite_axisgrid() #When you use pw.load_seagorngrid, 'overwrite_axisgrid' should be executed　in advance
            plot_joint_threshold_split(df_final, ['GC', 'YSC', 'NSC'], combo[0], combo[1], 'c27:Ngen',label_dict,  threshold_row=2,cat='sys',sample_size=10000, col_col='met', threshold_col=sorted([0.0002, 0.008, 0.0012, 0.3]),quantile=0.99, save=True)
            combo_list.append(combo)
        
        except Exception as e:
            print(f"Error plotting {combo[0]} vs {combo[1]}: {e}")
            continue
    else: 
        continue

## Classification

## Results